# 04 · Analyze a Failure / 失败分析
预计 20–30 分钟。区分事实（日志/画面显示什么）、原因假设（可能为什么）和改进方案。自动程序不把低成功率直接归因于某个组件。若本次全部成功，可选择同组此前失败或教师明确标记的失败记录；不把教师结果冒充自己的。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
EVALUATION = stage_path(RUN, "evaluation")
TRAINING = stage_path(RUN, "training")

In [ ]:
results = json.loads((EVALUATION / "summary.json").read_text())
failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"]]
for r in failures:
    print(r["episode_id"], "environment success:", r["environment_success"], "steps:", r["steps"])
if not failures:
    teacher = ROOT / "checkpoints/reference_evaluation/summary.json"
    if not teacher.exists():
        raise RuntimeError("本次没有可分析失败；请教师提供带来源的真实失败记录。")
    EVALUATION = teacher.parent
    results = json.loads(teacher.read_text())
    failures = [r for r in results["episodes"] if not r["stable_success"] and not r["error"]]
    if not failures:
        raise RuntimeError("教师参考也没有失败，请另选真实失败实验。")
    print("Failure evidence source: instructor reference; student evaluation unchanged")
EPISODE = failures[0]["episode_id"]  # 可改成上方任一真实失败ID
print("Selected:", EPISODE)

In [ ]:
from manipulation_lab.reporting import failure_evidence
ANALYSIS = next_stage(RUN, "analysis")
ANALYSIS.mkdir()
facts = failure_evidence(EVALUATION, EPISODE, ANALYSIS)
print(json.dumps(facts, indent=2))
display(Image(filename=str(ANALYSIS / "failure_signals.png")))
video = EVALUATION / (EPISODE + ".mp4")
display(Video(str(video), embed=True))
for i in range(3):
    display(Image(filename=str(ANALYSIS / f"keyframe_{i}.png")))

填写四题：1. Reaching / Grasping / Lifting / Transporting / Placing 哪个阶段失败？允许 unknown。2. 第一个可观察错误是什么、在哪个时间步？3. 可能原因是什么？4. 怎样改进策略或机器人系统，如何验证改进？阶段需要看轨迹，不能仅凭最后一帧。

In [ ]:
FAILURE_STAGE = ""
FIRST_OBSERVABLE_ERROR = ""
HYPOTHESIZED_CAUSE = ""
IMPROVEMENT = ""
write_json(ANALYSIS / "failure_analysis.json", {
    "episode": EPISODE, "evidence_directory": str(EVALUATION),
    "evidence_policy_role": results["policy_role"],
    "stage": FAILURE_STAGE, "first_observable_error": FIRST_OBSERVABLE_ERROR,
    "hypothesized_cause": HYPOTHESIZED_CAUSE, "improvement": IMPROVEMENT})
REPORT = next_stage(RUN, "report")
command("generate_report.py", "--run", RUN, "--output", REPORT)
display(Markdown((REPORT / "report.md").read_text()))
print("提交前填写空白答案，并将报告整理为1–2页；原始日志和checkpoint可另附。")

提交：demonstration_analysis.json、一份 1–2 页报告、真实训练曲线、评测 CSV/JSON、配置、关键帧。按理解与证据评分：20% / 25% / 30% / 25%；不以成功率高低为主要标准。